# Dataset Structure and Cleaning Audit

This section documents the raw structure of the Clark County Senior Food Insecurity Survey dataset before variable construction or formal analysis. The goal is to identify the basic dataset dimensions, column structure, data types, missingness patterns, and initial cleaning or recoding issues that need to be resolved before constructing analysis variables.

## Initial Data Load

The first step is to load the raw dataset from the shared `data/` folder using a relative path. Using a relative path keeps the notebook portable across team members' computers.

In [1]:
# Import core libraries for data inspection

import pandas as pd
import numpy as np

In [2]:
# Load raw dataset using relative path from notebooks/ to data/ and preview first rows

df = pd.read_csv("../data/Clark_County_Senior_FIS_Data_N_800.csv")

df.head()

,_survnum,ptype,qintro,qs1,qs2,qs3,qs4,qs5,q1,q1oth1,...,q51,q52,stopdate,pin,qs3xR1,q40xR1,zip,fips,tract,keycode
0,641,1,1,1,1,72.0,NaN,1,3,NaN,...,2,2,12/17/2019,23750322,NaN,NaN,89115,32003,6001,NaN
1,414,1,1,1,1,76.0,NaN,1,2,NaN,...,1,3,12/9/2019,21630873,NaN,NaN,89012,32003,5338,NaN
2,105,2,1,1,1,75.0,2.0,1,4,NaN,...,2,1,12/4/2019,19640354,NaN,NaN,89119,32003,2847,4.0
3,773,1,1,1,1,74.0,NaN,1,5,NaN,...,1,2,12/18/2019,21802416,NaN,NaN,89104,32003,1300,NaN
4,44,1,1,1,1,71.0,2.0,1,4,NaN,...,2,2,11/27/2019,18562023,NaN,NaN,89106,32003,301,NaN


The raw dataset was loaded from the shared `data/` folder using a relative path. The preview confirms that the dataset loads successfully and contains survey-style variables such as screener variables, demographic variables, food security questions, financial literacy variables, and location/date variables.

## Dataset Dimensions

The next step is to confirm the number of observations and columns in the raw dataset before any cleaning or variable construction.

In [3]:
# Check dataset dimensions: number of rows and columns

df.shape

(800, 112)

The raw dataset contains 800 observations and 112 columns. At this stage, no cleaning or variable construction has been performed.

## Column Names

The next step is to display the full list of column names to compare the dataset structure with the survey/codebook.

In [4]:
# Display all column names in a readable table

column_names = pd.DataFrame({
    "column_index": range(len(df.columns)),
    "column_name": df.columns
})

column_names

,column_index,column_name
0,0,_survnum
1,1,ptype
2,2,qintro
3,3,qs1
4,4,qs2
...,...,...
107,107,q40xR1
108,108,zip
109,109,fips
110,110,tract


The table above lists all 112 column names in their original dataset order. The column names mostly follow the survey/codebook structure, with variables labeled using question numbers such as `q1`, `q4`, `q17`, `q44`, and related variants.

## Data Types

The next step is to inspect how Python reads each variable. This helps identify which columns are numeric, which are stored as text/object variables, and whether any variables may need type conversion before analysis.

In [5]:
# Check variable names, non-null counts, and data types

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Columns: 112 entries, _survnum to keycode
dtypes: float64(39), int64(71), str(2)
memory usage: 700.1 KB


In [6]:
# Summarize the number of variables by data type

df.dtypes.value_counts()

int64      71
float64    39
str         2
Name: count, dtype: int64

In [7]:
# Identify object/string columns

df.select_dtypes(include="object").columns.tolist()

C:\Users\ruben\AppData\Local\Temp\ipykernel_29404\2444311330.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include="object").columns.tolist()


['q1oth1', 'stopdate']

The dataset contains 112 columns: 71 integer variables, 39 float variables, and 2 object/string variables. Most survey response variables are stored numerically. The two object/string variables are `q1oth1` and `stopdate`, which require closer inspection before deciding whether they need recoding or type conversion.

### Object Columns

Because only two variables are stored as object/string columns, we inspect them directly.

In [8]:
# Inspect object/string columns

df[["q1oth1", "stopdate"]].head(10)

,q1oth1,stopdate
0,NaN,12/17/2019
1,NaN,12/9/2019
2,NaN,12/4/2019
3,NaN,12/18/2019
4,NaN,11/27/2019
5,NaN,12/16/2019
6,NaN,12/19/2019
7,NaN,12/19/2019
8,NaN,12/7/2019
9,NaN,12/5/2019


In [9]:
# Check unique values / examples for object columns

for col in ["q1oth1", "stopdate"]:
    print(f"\n{col}")
    print(df[col].dropna().unique()[:20])


q1oth1
<StringArray>
['High school equivalent']
Length: 1, dtype: str

stopdate
<StringArray>
['12/17/2019',  '12/9/2019',  '12/4/2019', '12/18/2019', '11/27/2019',
 '12/16/2019', '12/19/2019',  '12/7/2019',  '12/5/2019', '12/11/2019',
 '12/20/2019',  '12/6/2019',  '12/8/2019', '12/10/2019', '11/26/2019']
Length: 15, dtype: str


The `stopdate` variable contains survey completion dates and may need to be converted to a datetime format if used. The `q1oth1` variable appears to be an open-ended “Other” response related to education and is mostly missing, so it is likely an auxiliary variable rather than a primary analysis variable.

## Missing Values

Next, we summarize missingness across all variables. This helps identify variables with substantial nonresponse, structural missingness from survey skip patterns, or variables that may not be useful for analysis.

In [10]:
# Create combined missingness summary by count and percentage

missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100
}).sort_values("missing_percent", ascending=False)

missing_summary.head(25)

,missing_count,missing_percent
q2R6,800,100.000
q2R4,800,100.000
q2R5,800,100.000
q6R6,800,100.000
q6R9,800,100.000
q6R10,800,100.000
q6R12,800,100.000
q6R11,800,100.000
q6R5,800,100.000
q6R8,800,100.000


The table above summarizes missingness by variable, showing both the number and percentage of missing observations. Several variables have high missingness, including some variables that are completely missing. This does not necessarily mean the data are unusable; some missingness may reflect survey skip patterns, multi-response variables, or open-ended follow-up questions that only apply to a small number of respondents.

At this stage, missing values are being documented rather than cleaned or dropped. Before variable construction, we need to distinguish between true nonresponse, structural missingness from skip logic, and variables that may not be analytically useful.

## Codebook Match and Variable Map

Based on the column-name table above and the codebook, the dataset appears to follow the expected survey/question structure. The variables can be grouped approximately as follows:

- Survey/admin variables: `_survnum`, `ptype`, `qintro`, `stopdate`, `pin`, `zip`
- Screener variables: `qs1`, `qs2`, `qs3`, `qs4`, `qs5`
- Demographics: `q1`, `q1oth1`, `q2R*`, `q3`, `q4`
- Household composition: `q5`, `q6R*`, `q7`
- Food spending / meal gap questions: `q8`–`q16`
- Food security questions: `q17`–`q22`
- Health/disability variables: `q23`–`q30`
- Housing variables: `q31`–`q33`
- Social support variables: `q34`–`q37`
- Income sources / SNAP variables: `q38*`, `q39`, `q40`
- Spending tradeoffs: `q41a`–`q41d`
- Neighborhood/access/aging-in-place variables: `q42a`–`q42e`
- Financial literacy/capability variables: `q43`, `q44`, `q45*`
- Transportation variables: `q46`–`q48`
- Employment / mobility variables: `q49`–`q52`

## Response Coding Audit

The next step is to inspect raw response codes for selected representative variables across major survey sections. This helps identify how survey responses are coded, whether special responses such as “Unsure,” “Don’t Know,” or “Refused” appear in the data, and which variables may require recoding before analysis.

In [11]:
# Create value-count summary for representative variables

representative_vars = [
    "qs1", "qs2", "qs3", "qs4", "qs5",
    "q1", "q3", "q4",
    "q17", "q18", "q19", "q21", "q22",
    "q23", "q31", "q39",
    "q41a", "q42a",
    "q44", "q45b", "q45c"
]

coding_summary = []

for col in representative_vars:
    counts = df[col].value_counts(dropna=False).sort_index()
    for value, count in counts.items():
        coding_summary.append({
            "variable": col,
            "value": value,
            "count": count,
            "percent": round(count / len(df) * 100, 2)
        })

coding_summary = pd.DataFrame(coding_summary)

coding_summary

,variable,value,count,percent
0,qs1,1.0,800,100.00
1,qs2,1.0,800,100.00
2,qs3,60.0,14,1.75
3,qs3,61.0,21,2.62
4,qs3,62.0,23,2.88
...,...,...,...,...
117,q45c,2.0,97,12.12
118,q45c,3.0,27,3.38
119,q45c,4.0,92,11.50
120,q45c,5.0,174,21.75


The response-coding table summarizes raw values, counts, and percentages for representative variables across major survey sections. This table is intended as an initial audit only. Final interpretation of these values requires matching each variable to its codebook definitions before recoding or constructing analysis variables.

## Duplicate and Identifier Checks

This section checks whether the dataset contains fully duplicated rows and whether likely administrative identifiers appear more than once. The goal is not to remove observations yet, but to flag whether the raw dataset may contain repeated records that need to be resolved before analysis.

In [12]:
# Check for fully duplicated rows

df.duplicated().sum()

np.int64(245)

In [13]:
# Check whether potential ID/admin variables are unique

id_vars = ["_survnum", "pin"]

for col in id_vars:
    print(f"\n{col}")
    print("Unique values:", df[col].nunique(dropna=False))
    print("Duplicated values:", df[col].duplicated().sum())


_survnum
Unique values: 555
Duplicated values: 245

pin
Unique values: 555
Duplicated values: 245


In [14]:
# Inspect examples of duplicated rows

duplicate_rows = df[df.duplicated(keep=False)].sort_values(["_survnum", "pin"])

duplicate_rows.head(10)

,_survnum,ptype,qintro,qs1,qs2,qs3,qs4,qs5,q1,q1oth1,...,q51,q52,stopdate,pin,qs3xR1,q40xR1,zip,fips,tract,keycode
147,4,1,1,1,1,81.0,2.0,1,2,NaN,...,1,2,11/26/2019,18661345,NaN,NaN,89029,32003,5705,NaN
639,4,1,1,1,1,81.0,2.0,1,2,NaN,...,1,2,11/26/2019,18661345,NaN,NaN,89029,32003,5705,NaN
40,31,1,1,1,1,77.0,2.0,1,2,NaN,...,1,2,11/27/2019,18863133,NaN,NaN,89122,32003,5006,NaN
557,31,1,1,1,1,77.0,2.0,1,2,NaN,...,1,2,11/27/2019,18863133,NaN,NaN,89122,32003,5006,NaN
70,33,1,1,1,1,84.0,2.0,1,2,NaN,...,2,2,11/27/2019,18834740,NaN,NaN,89108,32003,3423,NaN
211,33,1,1,1,1,84.0,2.0,1,2,NaN,...,2,2,11/27/2019,18834740,NaN,NaN,89108,32003,3423,NaN
659,33,1,1,1,1,84.0,2.0,1,2,NaN,...,2,2,11/27/2019,18834740,NaN,NaN,89108,32003,3423,NaN
759,33,1,1,1,1,84.0,2.0,1,2,NaN,...,2,2,11/27/2019,18834740,NaN,NaN,89108,32003,3423,NaN
168,34,1,1,1,1,82.0,2.0,1,4,NaN,...,2,3,11/27/2019,18735110,NaN,NaN,89103,32003,2954,NaN
317,34,1,1,1,1,82.0,2.0,1,4,NaN,...,2,3,11/27/2019,18735110,NaN,NaN,89103,32003,2954,NaN


The dataset contains 245 fully duplicated rows. The potential administrative variables `_survnum` and `pin` each contain 555 unique values and 245 duplicated values, which is consistent with the duplicate-row count. This suggests that some observations may be repeated exactly in the raw dataset.

At this stage, no duplicate observations are being dropped. This issue should be flagged for the team before deciding whether duplicate rows should be removed or whether they reflect some intended feature of the data.

In [15]:
# Count how often each survey number appears

survnum_counts = df["_survnum"].value_counts()

survnum_counts.head(10)

_survnum
109    5
586    5
534    4
33     4
472    4
173    4
851    4
669    4
414    3
105    3
Name: count, dtype: int64

In [16]:
# Summarize duplicate frequency among survey numbers

survnum_counts.value_counts().sort_index()

count
1    371
2    133
3     43
4      6
5      2
Name: count, dtype: int64

In [17]:
# Count how often each pin appears

pin_counts = df["pin"].value_counts()

pin_counts.head(10)

pin
18974651    5
23557277    5
21554849    4
18834740    4
21920923    4
18660692    4
21945275    4
26498453    4
21630873    3
19640354    3
Name: count, dtype: int64

In [18]:
# Summarize duplicate frequency among pins

pin_counts.value_counts().sort_index()

count
1    371
2    133
3     43
4      6
5      2
Name: count, dtype: int64

The duplicate-row inspection shows that 245 rows are exact duplicates in the raw dataset. Both `_survnum` and `pin` have 555 unique values, with some identifiers appearing up to five times. This suggests that some observations are repeated in the raw data rather than merely sharing an ID value.

At this stage, no duplicate rows are being dropped. This issue should be flagged before cleaning because removing duplicates would change the analytic sample size from 800 observations to 555 unique records.

## Person 1 Summary / Handoff

This audit documents the raw dataset structure before variable construction. The dataset contains 800 rows and 112 columns. Most variables are numeric survey-response variables, with two object columns: `q1oth1` and `stopdate`.

Key issues identified for the team include high missingness in several variables, the need to distinguish true missingness from survey skip-pattern missingness, and 245 fully duplicated rows. No observations or variables were dropped in this audit.

Next steps should include variable-specific review of the food security questions, financial literacy/capability questions, and codebook-based decisions about missing values, special response categories, and duplicate handling.

---
## Person 2: Food Insecurity Questions Audit
    - Locate the food security variables
    - Check response coding
    - Identify missing/unsure/refused variables
    - Summarize what will likely be needed to build the dependent variable later


Locating the food security variables: These are the 6 USDA questions from the survey. q17-q22. 

I will be creating an affirmative indicator for each of the 6 questions. An "affirmative" means the person expressed food hardship on that question. Per USDA rules:

In [19]:
# Drop exact duplicate rows identified in Person 1 audit
# 245 fully duplicated rows were flagged; dropping them reduces the 
# analytic sample from 800 to 555 unique records.

df = df.drop_duplicates()
print(f"Rows after dropping duplicates: {len(df)}")

Rows after dropping duplicates: 555


I decided to drop the duplicated columnes for the time being. We can always decide to investigate further and add them but these are the 555 unique rows (no duplicates).

In [20]:
# Audit raw response codes for all 6 USDA food security variables.
# This confirms what values actually appear in the data before any recoding.

fs_vars = ['q17', 'q18', 'q19', 'q20', 'q21', 'q22']

for v in fs_vars:
    print(f"--- {v} ---")
    print(df[v].value_counts(dropna=False).sort_index())
    print(f"NaN count: {df[v].isna().sum()}")
    print()

--- q17 ---
q17
1     63
2    117
3    359
4     16
Name: count, dtype: int64
NaN count: 0

--- q18 ---
q18
1     83
2    120
3    333
4     19
Name: count, dtype: int64
NaN count: 0

--- q19 ---
q19
1     88
2    461
3      6
Name: count, dtype: int64
NaN count: 0

--- q20 ---
q20
1.0     42
2.0     30
3.0     15
4.0      1
NaN    467
Name: count, dtype: int64
NaN count: 467

--- q21 ---
q21
1     84
2    460
3     11
Name: count, dtype: int64
NaN count: 0

--- q22 ---
q22
1     62
2    489
3      4
Name: count, dtype: int64
NaN count: 0



**q17** (Food didn't last): 1=Often (63), 2=Sometimes (117), 3=Never (359), 4=Unsure (16). No missing.  
**q18** (Couldn't afford balanced meals): 1=Often (83), 2=Sometimes (120), 3=Never (333), 4=Unsure (19). No missing.  
**q19** (Cut or skipped meals): 1=Yes (88), 2=No (461), 3=Unsure (6). No missing.  
**q20** (How often — follow-up to q19): 1=Almost every month (42), 2=Some months (30), 
3=Only 1-2 months (15), 4=Unsure (1), NaN=467. NaN is expected — see skip logic cell below.  
**q21** (Ate less than should): 1=Yes (84), 2=No (460), 3=Unsure (11). No missing.  
**q22** (Hungry but didn't eat): 1=Yes (62), 2=No (489), 3=Unsure (4). No missing.

In [21]:
# q20 is a follow-up question only asked when q19 = 1 (Yes).
# All other respondents should have NaN for q20.
# This cross-tab verifies the skip pattern is clean in the data.

skip_check = pd.crosstab(
    df['q19'].astype(str),
    df['q20'].isna(),
    rownames=['q19 (1=Yes, 2=No, 3=Unsure)'],
    colnames=['q20 is NaN']
)

print(skip_check)

q20 is NaN                   False  True 
q19 (1=Yes, 2=No, 3=Unsure)              
1                               88      0
2                                0    461
3                                0      6



Looking at the "skip logic" for q20 here. Since it depends on a "yes" from q19 (88 in total), we need to make sure that there are as many responses on q20 as there are "yes" responses in q19.

The skip logic is clean. All 88 respondents who answered Yes to q19 have a valid response for q20 (False = not missing). 

All 461 who said No and all 6 who said Unsure have NaN for q20 (True = missing). 

No records violate the expected skip pattern.

In [22]:
# Count "Unsure" responses across all 6 variables.
# Unsure is coded as 4 for q17/q18/q20, and as 3 for q19/q21/q22.

unsure_map = {'q17': 4, 'q18': 4, 'q19': 3, 'q20': 4, 'q21': 3, 'q22': 3}

print("Unsure response counts:")
for var, code in unsure_map.items():
    n = (df[var] == code).sum()
    print(f"  {var}: {n} respondents")

Unsure response counts:
  q17: 16 respondents
  q18: 19 respondents
  q19: 6 respondents
  q20: 1 respondents
  q21: 11 respondents
  q22: 4 respondents


**Coding decision:** "Unsure" responses are treated as non-affirmative (coded 0) 
in the food security scale. This follows USDA guidance, which treats "Don't Know" 
and "Refused" as non-affirmative. The counts are small (1–19 per variable out of 555), 
so the impact on classification is minimal. This decision is documented here for 
transparency and replication.

In [23]:
# Create a binary affirmative indicator for each of the 6 USDA questions.
# Affirmative = the respondent expressed food hardship on that item.
# Unsure and Never/No responses are coded 0 (non-affirmative).
# NaN on q20 (from skip logic) is filled with 0 — not affirmed.

aff_cols = ['aff_q17','aff_q18','aff_q19','aff_q20','aff_q21','aff_q22']

affirmatives = pd.DataFrame({
    'aff_q17': df['q17'].isin([1, 2]).astype(int),   # Often or Sometimes
    'aff_q18': df['q18'].isin([1, 2]).astype(int),   # Often or Sometimes
    'aff_q19': (df['q19'] == 1).astype(int),          # Yes
    'aff_q20': df['q20'].isin([1, 2]).fillna(0).astype(int),  # Almost every/Some months; NaN→0
    'aff_q21': (df['q21'] == 1).astype(int),          # Yes
    'aff_q22': (df['q22'] == 1).astype(int),          # Yes
})

df = pd.concat([df, affirmatives], axis=1)

# Confirm affirmative counts
print("Affirmative counts per item:")
for col in aff_cols:
    print(f"  {col}: {df[col].sum()} ({df[col].mean()*100:.1f}%)")

Affirmative counts per item:
  aff_q17: 180 (32.4%)
  aff_q18: 203 (36.6%)
  aff_q19: 88 (15.9%)
  aff_q20: 72 (13.0%)
  aff_q21: 84 (15.1%)
  aff_q22: 62 (11.2%)


In [24]:
# Sum the 6 affirmative indicators into a raw score (0–6).
# This is the household's total count of food hardship items affirmed.

df['raw_score'] = df[aff_cols].sum(axis=1)

print("Raw score distribution:")
print(df['raw_score'].value_counts().sort_index())
print(f"\nMean raw score: {df['raw_score'].mean():.3f}")

Raw score distribution:
raw_score
0    300
1     93
2     59
3     23
4     25
5     21
6     34
Name: count, dtype: int64

Mean raw score: 1.241


C:\Users\ruben\AppData\Local\Temp\ipykernel_29404\169058088.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['raw_score'] = df[aff_cols].sum(axis=1)


In [25]:
# Apply USDA classification to create the binary dependent variable.
# Raw score 0–1 → Food Secure  (food_insecure = 0)
# Raw score 2–6 → Food Insecure (food_insecure = 1)
# Source: USDA Six-Item Food Security Scale scoring guide (ERS, May 2024)

df['food_insecure'] = (df['raw_score'] >= 2).astype(int)

n_insecure = df['food_insecure'].sum()
n_secure   = (df['food_insecure'] == 0).sum()

print(f"Food Insecure (raw score 2–6): {n_insecure} ({n_insecure/len(df)*100:.1f}%)")
print(f"Food Secure   (raw score 0–1): {n_secure}  ({n_secure/len(df)*100:.1f}%)")

Food Insecure (raw score 2–6): 162 (29.2%)
Food Secure   (raw score 0–1): 393  (70.8%)


C:\Users\ruben\AppData\Local\Temp\ipykernel_29404\73709651.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['food_insecure'] = (df['raw_score'] >= 2).astype(int)


## Person 2 Summary / Handoff

The six USDA food security variables (q17–q22) map cleanly to the survey instrument 
with no unexpected codes or missing values beyond the expected skip pattern on q20.

**Key decisions made:**
- "Unsure" responses treated as non-affirmative (coded 0), consistent with USDA guidance.
- q20 NaN values (respondents who skipped due to q19=No) treated as 0 — not affirmed.
- Duplicate rows were dropped prior to variable construction (555 unique records).

**Dependent variable constructed:**
- `raw_score`: sum of affirmative responses across 6 items (range 0–6)
- `food_insecure`: binary variable; 1 = food insecure (raw score ≥ 2), 0 = food secure

**Result:** 162 households (29.2%) are classified as food insecure. This is notably 
higher than national senior averages (~15%), which underscores the relevance of this 
study for Clark County stakeholders.

**Next steps:** Carlos's section will audit and construct the financial literacy 
explanatory variables. The `food_insecure` variable in `df` is ready for regression once 
those variables are built.

---
## Person 3 (Carlos): Financial Literacy/Capability Variables Audit

This section is my part of the audit. Based on the variable map from Chad's section and what I can piece together from the column names, the financial literacy and capability variables appear to be `q43`, `q44`, and `q45a` through `q45h`. My goal here is to look at the raw response codes, flag anything that looks off or ambiguous, check for missingness, and lay out what the team will need to sort out before we can actually build the explanatory variables for the regression.

### Locating the Financial Literacy Variables

As far as I can tell, there are three groups of variables that relate to financial literacy and capability:

- `q43`: This looks like a financial knowledge score. It ranges from 0 to 5, which, to my understanding, suggests it counts how many questions a respondent answered correctly on some kind of short financial knowledge quiz.
- `q44`: This one seems to be a self-rated financial capability measure on a 1-5 scale, like the respondent's own sense of how well they manage their finances.
- `q45a` through `q45h`: A battery of 8 items covering financial practices like budgeting, tracking spending, and paying bills on time. Each item uses a 6-point scale where **1 = "I do not have the means to do this"** and **6 = "I am doing this all of the time."** Higher values mean better financial behavior. No reverse coding needed.

In [26]:
# Audit raw response codes for all financial literacy/capability variables.
# This mirrors the approach used in the Person 2 food security audit.

fin_vars = ['q43', 'q44', 'q45a', 'q45b', 'q45c', 'q45d', 'q45e', 'q45f', 'q45g', 'q45h']

for v in fin_vars:
    print(f'--- {v} ---')
    print(df[v].value_counts(dropna=False).sort_index())
    print(f'NaN count: {df[v].isna().sum()}')
    print()

--- q43 ---
q43
0     94
1     38
2     32
3     50
4    102
5    239
Name: count, dtype: int64
NaN count: 0

--- q44 ---
q44
1     19
2     22
3     53
4    125
5    336
Name: count, dtype: int64
NaN count: 0

--- q45a ---
q45a
1     80
2    100
3     25
4     71
5    116
6    163
Name: count, dtype: int64
NaN count: 0

--- q45b ---
q45b
1     57
2    116
3     24
4     68
5    114
6    176
Name: count, dtype: int64
NaN count: 0

--- q45c ---
q45c
1     19
2     65
3     21
4     58
5    123
6    269
Name: count, dtype: int64
NaN count: 0

--- q45d ---
q45d
1      8
2     37
3     12
4     64
5     85
6    349
Name: count, dtype: int64
NaN count: 0

--- q45e ---
q45e
1     11
2     21
3      7
4     24
5     69
6    423
Name: count, dtype: int64
NaN count: 0

--- q45f ---
q45f
1     21
2     28
3      9
4     55
5     79
6    363
Name: count, dtype: int64
NaN count: 0

--- q45g ---
q45g
1     29
2     78
3     15
4    112
5    100
6    221
Name: count, dtype: int64
NaN count: 0

--- q

**What I'm seeing from the raw audit:**

- `q43`: Goes from 0 to 5 with zero missing values. The fact that it starts at 0 (not 1) is consistent with my read that it's a count of correct answers rather than a rating scale. All 555 respondents have a value here, which is good.
- `q44`: Goes from 1 to 5, also no missing. The distribution is pretty skewed toward 5 -- around 60.5% of respondents are at the top. That's a bit surprising but could just mean most seniors feel fairly confident about managing their money.
- `q45a` through `q45h`: All range from 1 to 6 with no missing values, which is clean. The issue is that value 6 is by far the most common response on every single item -- anywhere from about 29% on q45a to 76% on q45e. That's too dominant to be a normal scale endpoint. My best guess is that 6 means something like "Does not apply" or "Never," but I can't confirm that without the codebook. This is the main thing that needs to be resolved before we can use these variables.

### Taking a Closer Look at Value 6 on the q45 Battery

I wanted to dig a little deeper into the value 6 issue before just flagging it and moving on. My thinking was: if 6 really means "Does not apply," then we'd expect some respondents to answer 6 on every single item, basically saying none of these financial hardship situations ever apply to them. So I checked how many respondents did exactly that, and how the counts of 6s per person are distributed overall.

In [27]:
# Check how many respondents have value 6 on all 8 q45 items.
# Also summarize the distribution of how many items equal 6 per respondent.

q45_vars = ['q45a', 'q45b', 'q45c', 'q45d', 'q45e', 'q45f', 'q45g', 'q45h']

# Count how many q45 items equal 6 for each respondent
count_sixes = (df[q45_vars] == 6).sum(axis=1)

print('Distribution of number of q45 items = 6 per respondent:')
print(count_sixes.value_counts().sort_index())
print()
print(f'Respondents with ALL 8 items = 6: {(count_sixes == 8).sum()}')
print(f'Respondents with at least one item != 6: {(count_sixes < 8).sum()}')

Distribution of number of q45 items = 6 per respondent:
0    54
1    42
2    62
3    61
4    82
5    73
6    76
7    58
8    47
Name: count, dtype: int64

Respondents with ALL 8 items = 6: 47
Respondents with at least one item != 6: 508


So 47 respondents answered 6 on all 8 items. Now that we have the codebook, this makes complete sense -- value 6 means **"I am doing this all of the time,"** the highest engagement level. Items like paying bills on time (q45e, 76%) and reviewing bills (q45d, 63%) naturally skew toward 6 because most seniors consistently do these things.

**Value 6 is resolved:** It is a valid, meaningful response and should be kept as-is in any composite score. No special treatment needed.

The one remaining open question is **value 1** -- "I do not have the means to do this" -- which signals a financial constraint rather than a behavioral choice, and is conceptually different from values 2–6. The team should decide whether to keep it as the lowest scale point or treat it separately before building the composite.

In [28]:
# Build a percentage distribution table for the q45 battery (values 1-6).
# Each row is a response value; each column is a q45 item.

pct_table = pd.DataFrame({
    v: df[v].value_counts(normalize=True).sort_index() * 100
    for v in q45_vars
}).round(1)

pct_table.index.name = 'value'
print('Response distribution (%) for q45 battery:')
pct_table

Response distribution (%) for q45 battery:


,q45a,q45b,q45c,q45d,q45e,q45f,q45g,q45h
value,,,,,,,,
1,14.4,10.3,3.4,1.4,2.0,3.8,5.2,9.0
2,18.0,20.9,11.7,6.7,3.8,5.0,14.1,10.1
3,4.5,4.3,3.8,2.2,1.3,1.6,2.7,2.5
4,12.8,12.3,10.5,11.5,4.3,9.9,20.2,8.3
5,20.9,20.5,22.2,15.3,12.4,14.2,18.0,13.2
6,29.4,31.7,48.5,62.9,76.2,65.4,39.8,56.9


### Response Distribution for the q45 Battery

Just to make it easier to see the pattern across all 8 items, I put together a percentage table. You can see pretty clearly how value 6 dominates some items much more than others.

### Missingness Check

One thing that was actually reassuring here -- none of the financial literacy variables have any missing values across the 555 deduplicated records. So at least we won't have to deal with imputation or dropping observations on this front. That should make the variable construction step a bit cleaner once we sort out the value 6 question.

In [29]:
# Confirm zero missing values across all financial literacy/capability variables.

missing_fin = pd.DataFrame({
    'missing_count': df[fin_vars].isna().sum(),
    'missing_percent': (df[fin_vars].isna().mean() * 100).round(2)
})

print('Missingness summary for financial literacy variables:')
missing_fin

Missingness summary for financial literacy variables:


,missing_count,missing_percent
q43,0,0.0
q44,0,0.0
q45a,0,0.0
q45b,0,0.0
q45c,0,0.0
q45d,0,0.0
q45e,0,0.0
q45f,0,0.0
q45g,0,0.0
q45h,0,0.0


## Person 3 Summary / Handoff

All 10 financial literacy/capability variables (`q43`, `q44`, `q45a` through `q45h`) are fully observed across all 555 records -- no missing values anywhere, which is a nice clean starting point.

**Main takeaways:**
- `q43`: Financial knowledge score, 0-5, no missing. Looks ready to use as-is or potentially recoded into groups.
- `q44`: Self-rated financial capability, 1-5, no missing. Heavily skewed toward 5, but that might just reflect the sample.
- `q45a` through `q45h`: 8-item financial behavior battery, 1–6 scale, no missing. Scale is fully resolved -- value 6 means "I am doing this all of the time" (highest engagement), not a special category. Higher = better financial behavior, no reverse coding needed. Only open question is how to treat value 1 ("I do not have the means"), which signals financial constraint rather than behavioral choice.

**Things the team needs to decide before we build variables:**
1. What does value 6 mean on the q45 items? (Codebook question -- can't move forward without this)
2. Do we use q43, q44, and q45 as separate regressors, or do we try to combine some of them into a composite financial literacy index?
3. Which direction does the scale run -- do higher values mean more capable/less distressed, or the other way around? This matters for how we interpret regression coefficients.

Once those are settled, we should be in good shape to build the explanatory variables and run the logit/probit with Ruben's `food_insecure` variable as the outcome.